# PART 1 : Build The Dataset

In [166]:
import torch
import torch.nn as nn

In [167]:
text = """
hello world
hello transformers
transformers are amazing
"""

## Step 1 : Create Vocabulary

In [168]:
chars = sorted(list(set(text)))

print(chars)
print(len(chars))

['\n', ' ', 'a', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'r', 's', 't', 'w', 'z']
18


## Step 2 : Create Character → Integer Mapping

In [169]:
stoi = {
    ch:i
    for i,ch in enumerate(chars)
}

itos = {
    i:ch
    for i,ch in enumerate(chars)
}

print(stoi['h'])

print(itos[5])

7
f


# Step 3 : Encode Entire Corpus

In [170]:
encode = lambda s: [
    stoi[c]
    for c in s
]
print(
    encode("hello")
)

decode = lambda ids: ''.join(
    [
        itos[i]
        for i in ids
    ]
)
print(
    decode(
        encode("hello")
    )
)

[7, 4, 9, 9, 12]
hello


# Step 4 : Convert Whole Corpus

In [171]:
data = torch.tensor(
    encode(text),
    dtype=torch.long
)
print(data)
print(data.shape)

print(data[:50])

tensor([ 0,  7,  4,  9,  9, 12,  1, 16, 12, 13,  9,  3,  0,  7,  4,  9,  9, 12,
         1, 15, 13,  2, 11, 14,  5, 12, 13, 10,  4, 13, 14,  0, 15, 13,  2, 11,
        14,  5, 12, 13, 10,  4, 13, 14,  1,  2, 13,  4,  1,  2, 10,  2, 17,  8,
        11,  6,  0])
torch.Size([57])
tensor([ 0,  7,  4,  9,  9, 12,  1, 16, 12, 13,  9,  3,  0,  7,  4,  9,  9, 12,
         1, 15, 13,  2, 11, 14,  5, 12, 13, 10,  4, 13, 14,  0, 15, 13,  2, 11,
        14,  5, 12, 13, 10,  4, 13, 14,  1,  2, 13,  4,  1,  2])


# Step 5 : Train / Validation Split

In [172]:
n = int(
    0.9 * len(data)
)

train_data = data[:n]

val_data = data[n:]

# Step 6 : Create Context Window

This is where GPT training begins.

Suppose:

block_size = 8

Meaning:

Model sees at most 8 previous tokens

For one training sample:

Input:

h e l l o

Target:

e l l o _

Notice:

Every target is:

Next Character

# Step 7 : Build One Sample

In [173]:
block_size = 8

x = train_data[:block_size]

y = train_data[1:block_size+1]

print(x)
print(y)

for t in range(block_size):

    context = x[:t+1]

    target = y[t]

    print(
        context,
        "->",
        target
    )
    
    
#     h -> e

# he -> l

# hel -> l

# hell -> o

# hello -> space

# Stop here.

# At this point we have completed:

# ✓ Vocabulary
# ✓ Tokenization
# ✓ Encoding
# ✓ Decoding
# ✓ Dataset
# ✓ Context Window
# ✓ Next Token Targets

tensor([ 0,  7,  4,  9,  9, 12,  1, 16])
tensor([ 7,  4,  9,  9, 12,  1, 16, 12])
tensor([0]) -> tensor(7)
tensor([0, 7]) -> tensor(4)
tensor([0, 7, 4]) -> tensor(9)
tensor([0, 7, 4, 9]) -> tensor(9)
tensor([0, 7, 4, 9, 9]) -> tensor(12)
tensor([ 0,  7,  4,  9,  9, 12]) -> tensor(1)
tensor([ 0,  7,  4,  9,  9, 12,  1]) -> tensor(16)
tensor([ 0,  7,  4,  9,  9, 12,  1, 16]) -> tensor(12)


# PART 2 : Batch Generation

## Step 1 : Define Hyperparameters

Meaning:

block_size = 8

Model can look at at most 8 previous tokens.

batch_size = 4

Train on 4 sequences simultaneously.

In [174]:
block_size = 8
batch_size = 4

## Step 2 : Create get_batch()

Random Starting Positions
ix = torch.randint(
    len(data)-block_size,
    (batch_size,)
)

Suppose:

batch_size = 4

Output:

tensor([12,35,5,42])

Meaning:

Sample 1 starts at 12

Sample 2 starts at 35

Sample 3 starts at 5

Sample 4 starts at 42

In [175]:
def get_batch(split):

    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack(
        [
            data[i:i+block_size]
            for i in ix
        ]
    )

    y = torch.stack(
        [
            data[i+1:i+block_size+1]
            for i in ix
        ]
    )
    return x, y
# Every target is shifted by one position.



### test

In [176]:
xb, yb = get_batch("train")

print(xb.shape)
print(yb.shape)

torch.Size([4, 8])
torch.Size([4, 8])


# PART 3 : Token Embeddings

## Step 1

Create embedding layer.


In [177]:
vocab_size = 20
embed_dim = 32

token_embedding_table = nn.Embedding(
    vocab_size,
    embed_dim
)

## Step 2

Pass batch through embedding layer.

In [178]:
token_embeddings = token_embedding_table(
    xb
)

# PART 4 : Position Embeddings

## Step 1 : Create Position Embedding Table

In [179]:
position_embedding_table = nn.Embedding(
    block_size,
    embed_dim
)

# Visual
# Position 0 → [0.2, -0.5, ...]

# Position 1 → [1.1, 0.3, ...]

# Position 2 → [-0.7, 0.8, ...]

# ...

# These are trainable.

# GPT learns them during training.

## Step 2 : Create Position Indices

In [180]:
positions = torch.arange(block_size)

## Step 3 : Get Position Embeddings

In [181]:
position_embeddings = position_embedding_table(
    positions
)

## Step 4 : Combine Token + Position Embeddings

In [182]:
token_embeddings.shape
x = token_embeddings + position_embeddings

# PyTorch automatically broadcasts:

# (8,32)

# across all batches.

print(token_embeddings.shape)

print(position_embeddings.shape)

print(x.shape)

torch.Size([4, 8, 32])
torch.Size([8, 32])
torch.Size([4, 8, 32])


At this point:

Raw Tokens

Token Embedding

Position Embedding

Add

GPT Input Tensor

We finally have the actual input that enters GPT.

# PART 5 : Build Single Head Causal Attention


In [183]:
import math
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
class Head(nn.Module):

    def __init__(
        self,
        embed_dim,
        head_size,
        block_size
    ):

        super().__init__()

        self.key = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    block_size,
                    block_size
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        K = self.key(x)

        Q = self.query(x)

        V = self.value(x)

        scores = (
            Q @ K.transpose(-2,-1)
        ) / math.sqrt(K.shape[-1])

        scores = scores.masked_fill(
            self.tril[:T,:T] == 0,
            float("-inf")
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = weights @ V

        return out

In [184]:
head = Head(
    embed_dim=32,
    head_size=8,
    block_size=8
)

out = head(x)

print(out.shape)

torch.Size([4, 8, 8])


# Step 1 : Create MultiHeadAttention

In [185]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        num_heads,
        head_size,
        embed_dim,
        block_size
    ):

        super().__init__()

        self.heads = nn.ModuleList(
            [
                Head(
                    embed_dim,
                    head_size,
                    block_size
                )
                for _ in range(num_heads)
            ]
        )

        self.proj = nn.Linear(
            embed_dim,
            embed_dim
        )

    def forward(self, x):

        out = torch.cat(
            [
                h(x)
                for h in self.heads
            ],
            dim=-1
        )

        out = self.proj(out)

        return out

In [186]:
mha = MultiHeadAttention(
    num_heads=4,
    head_size=8,
    embed_dim=32,
    block_size=8
)

out = mha(x)

print(out.shape)

torch.Size([4, 8, 32])


# PART 6 : Feed Forward Network (nanoGPT Style)

## Step 1 : Create FeedForward Class

In [187]:
class FeedForward(nn.Module):

    def __init__(
        self,
        embed_dim
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                embed_dim,
                4 * embed_dim
            ),

            nn.ReLU(),

            nn.Linear(
                4 * embed_dim,
                embed_dim
            )

        )

    def forward(self, x):

        return self.net(x)

# PART 7 : GPT Block

## Step 1 : Constructor

In [188]:
class Block(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        block_size
    ):

        super().__init__()

        head_size = (
            embed_dim // num_heads
        )

        self.sa = MultiHeadAttention(
            num_heads,
            head_size,
            embed_dim,
            block_size
        )

        self.ffwd = FeedForward(
            embed_dim
        )

        self.ln1 = nn.LayerNorm(
            embed_dim
        )

        self.ln2 = nn.LayerNorm(
            embed_dim
        )

    def forward(self, x):

        x = x + self.sa(
            self.ln1(x)
        )

        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

At this point we have implemented the entire GPT decoder block:

LayerNorm

Masked MultiHeadAttention

Residual

LayerNorm

FeedForward

Residual

# PART 8 : GPTLanguageModel

Token Embeddings

Position Embeddings

GPT Block

GPT Block

GPT Block

Final LayerNorm

Language Model Head

Vocabulary Scores

## Step 1 : Create Class

In [189]:
class GPTLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        block_size,
        num_heads,
        num_layers
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            embed_dim
        )

        self.blocks = nn.Sequential(
            *[
                Block(
                    embed_dim,
                    num_heads,
                    block_size
                )
                for _ in range(num_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(
            embed_dim
        )

        self.lm_head = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(
                T,
                device=idx.device
            )
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(
                B * T,
                C
            )

            targets = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens
    ):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -self.block_size:]

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                [idx, next_token],
                dim=1
            )

        return idx

We have now completed:

✓ Dataset
✓ Tokenization
✓ get_batch()
✓ Embeddings
✓ Position Embeddings
✓ Attention Head
✓ MultiHeadAttention
✓ FeedForward
✓ GPT Block
✓ GPTLanguageModel

Now comes the most important stage:

Training

This is where the model actually learns.

# PART 10 : Instantiate Model

In [190]:
batch_size = 4
block_size = 8

embed_dim = 32

num_heads = 4

num_layers = 3

learning_rate = 1e-3

max_iters = 5000

Create model.

In [191]:
model = GPTLanguageModel(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    block_size=block_size,
    num_heads=num_heads,
    num_layers=num_layers
)

In [192]:
print(
    sum(
        p.numel()
        for p in model.parameters()
    )
)

39444


# PART 11 : Optimizer

Why model.parameters() ?

PyTorch automatically collects:

Embedding weights

Position embeddings

Q weights

K weights

V weights

FFN weights

LM Head weights

In [193]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

# PART 12 : First Forward Pass

In [194]:
xb, yb = get_batch("train")

logits, loss = model(
    xb,
    yb
)

print(logits.shape)

print(loss)

torch.Size([32, 20])
tensor(2.9610, grad_fn=<NllLossBackward0>)


# PART 13 : One Training Step

## Step 1

Reset gradients.

In [195]:
optimizer.zero_grad()

# Why?

# PyTorch accumulates gradients.

# Without this:

# Old gradients
# +
# New gradients

# would be added together.

##  Step 2

Backward Pass

In [196]:
loss.backward()

# What happens?

# PyTorch computes:

# ∂Loss/∂Parameters

# for:

# Embeddings

# Attention

# FFN

# LM Head

# Everything

## Step 3

Update Weights

In [197]:
optimizer.step()

# Formula conceptually:

# weight

# =

# weight

# -

# learning_rate × gradient

# One training step complete.

# PART 14 : Full Training Loop

Now repeat thousands of times.

In [198]:
for step in range(max_iters):

    xb, yb = get_batch("train")

    logits, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        print(
            step,
            loss.item()
        )
        
        
        
#         What Should Happen?

# Initially:

# loss ≈ 3.0

# or worse.

# After training:

# loss ≈ 2.0

# ↓

# 1.5

# ↓

# 1.0

# ↓

# 0.5

# depending on corpus size.

# Lower loss means:

# Better Next Token Prediction

0 2.898509979248047
100 0.7303407192230225
200 0.29974445700645447
300 0.35208332538604736
400 0.1842154711484909
500 0.19589301943778992
600 0.2746013402938843
700 0.2125329077243805
800 0.17212721705436707
900 0.21213562786579132
1000 0.20162293314933777
1100 0.08412548154592514
1200 0.1791720688343048
1300 0.29089421033859253
1400 0.13195523619651794
1500 0.08345936238765717
1600 0.1218290776014328
1700 0.14008116722106934
1800 0.1465996503829956
1900 0.13027890026569366
2000 0.18155547976493835
2100 0.13997851312160492
2200 0.31242021918296814
2300 0.07079844921827316
2400 0.2298334836959839
2500 0.04966671019792557
2600 0.11737647652626038
2700 0.15138331055641174
2800 0.12473644316196442
2900 0.1798982471227646
3000 0.2023484706878662
3100 0.11374998837709427
3200 0.18961158394813538
3300 0.17705659568309784
3400 0.08025596290826797
3500 0.1122966781258583
3600 0.13031694293022156
3700 0.1821916699409485
3800 0.05329326540231705
3900 0.07710570096969604
4000 0.11373982578516006
4

# PART 15 : Text Generation

Now we make GPT generate text.

Add this method inside:

Generate Text

Create starting token.

In [200]:
context = torch.zeros(
    (1,1),
    dtype=torch.long
)
generated_ids = model.generate(
    context,
    max_new_tokens=100
)
print(
    decode(
        generated_ids[0].tolist()
    )
)


hello transformers
transformers
transformers
transformers
transformers are amers
transformers
transf


Dataset

Tokenization

Embeddings

Position Embeddings

Masked Attention

Multi Head Attention

Feed Forward

GPT Blocks

Language Model Head

Cross Entropy Training

Autoregressive Generation